# Factors associated with altered psychomotor development in Chilean children

## Reproducible epidemiological analysis

This notebook contains **only the epidemiological arm** of the project. The machine-learning experiments from the exploratory notebook have been intentionally removed.

### Primary question
Which routinely recorded demographic, anthropometric, family-context, and vulnerability factors are associated with **altered psychomotor development** among children aged 0–60 months?

### Outcome
The source variable is `27.- Resultado del Desarrollo Psicomotor`.

- **Normal** → 0
- **Riesgo** or **Retraso** → 1 (`Altered DSM`)

### Main analysis
- Pooled 2019 and 2022 cohorts.
- Children aged 0–60 months.
- Multivariable logistic regression with HC3 robust standard errors.
- Age modeled non-linearly with a cubic B-spline (5 degrees of freedom).
- WHO-derived anthropometric categories are treated as categorical variables; no artificial numeric z-scores are created.
- A dedicated vulnerability analysis is performed because vulnerability recording has substantial and year-dependent missingness.

### Publication outputs
The notebook automatically exports publication-ready tables (`.xlsx` and `.csv`) and figures (`.png` and `.tiff`, 300 dpi) to the configured output directory.

## 1. Environment, paths, and reproducibility

The default paths match the original Google Drive project. For local/GitHub use, override them with the environment variables `DSM_PROJECT_DIR`, `DSM_DATA_PATH`, or `DSM_OUTPUT_DIR`.

In [ ]:
# Optional Google Drive mount (safe to skip outside Colab)
import os
import sys
import platform
import importlib
import importlib.metadata as importlib_metadata
from pathlib import Path

IN_COLAB = importlib.util.find_spec("google.colab") is not None
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)

DEFAULT_PROJECT_DIR = Path(
    "/content/drive/xxxxxxxxxxx"
)
PROJECT_DIR = Path(os.environ.get("DSM_PROJECT_DIR", DEFAULT_PROJECT_DIR))
DATA_PATH = Path(
    os.environ.get(
        "DSM_DATA_PATH",
        PROJECT_DIR / "control_sano_2019_2022.xlsx",
    )
)
OUTPUT_DIR = Path(
    os.environ.get(
        "DSM_OUTPUT_DIR",
        PROJECT_DIR / "xxxxxxxx" / "xxxxxx",
    )
)
FIGURE_DIR = OUTPUT_DIR / "figures"
TABLE_DIR = OUTPUT_DIR / "tables"
MODEL_DIR = OUTPUT_DIR / "model_outputs"

for directory in [OUTPUT_DIR, FIGURE_DIR, TABLE_DIR, MODEL_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Dataset not found: {DATA_PATH}\n"
        "Set DSM_DATA_PATH or DSM_PROJECT_DIR before running the notebook."
    )

print(f"Dataset: {DATA_PATH}")
print(f"Output directory: {OUTPUT_DIR}")

In [ ]:
# Package versions used in this run
import numpy as np
import pandas as pd
import scipy
import matplotlib
import statsmodels
import patsy
import openpyxl

packages = [
    "numpy",
    "pandas",
    "scipy",
    "matplotlib",
    "statsmodels",
    "patsy",
    "openpyxl",
]

versions = {
    "Python": sys.version.replace("\n", " "),
    "Platform": platform.platform(),
}
for package in packages:
    try:
        versions[package] = importlib_metadata.version(package)
    except importlib_metadata.PackageNotFoundError:
        versions[package] = "not installed"

version_table = pd.DataFrame(
    [(name, version) for name, version in versions.items()],
    columns=["Software", "Version"],
)
display(version_table)

version_table.to_csv(OUTPUT_DIR / "environment_versions.csv", index=False)
with open(OUTPUT_DIR / "environment_versions.txt", "w", encoding="utf-8") as handle:
    for name, version in versions.items():
        handle.write(f"{name}: {version}\n")

## 2. Imports and analysis configuration

Statistical significance is assessed using two-sided tests with `p < 0.05`. Individual coefficient tables also include Benjamini–Hochberg adjusted q-values as a multiplicity-aware sensitivity measure. Global Wald tests are preferred for multi-level categorical predictors.

In [ ]:
import re
import json
import warnings
from math import isfinite

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.stats import chi2_contingency, fisher_exact, mannwhitneyu
from patsy import bs
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.stats.multitest import multipletests
from statsmodels.stats.proportion import proportion_confint

warnings.filterwarnings("ignore", category=FutureWarning)

TARGET = "27.- Resultado del Desarrollo Psicomotor"
VALID_OUTCOMES = ["Normal", "Riesgo", "Retraso"]
AGE_MIN_MONTHS = 0
AGE_MAX_MONTHS = 60
RANDOM_SEED = 2026
DPI = 300

SOURCE_COLUMNS = {
    "age": "Edad de registro formulario",
    "sex": "Sexo",
    "height_for_age": "18.- Talla/Edad",
    "weight_for_height": "19.- Peso/Talla",
    "feeding": "42.- Alimentación",
    "parental_figure": "13.- Asiste al Control de Salud con Padre u otra imagen",
    "vulnerability": "33.- ¿Tiene Otra Vulnerabilidad?",
}

## 3. Helper functions

These functions perform source-data parsing, publication-table formatting, robust confidence intervals, and reproducible exports.

In [ ]:
def age_to_months(value):
    """Convert strings such as '2 años 3 meses 5 días' to decimal months."""
    if pd.isna(value):
        return np.nan
    text = str(value).strip()
    pattern = r"(\d+)\s*años?\s*(\d+)\s*meses?\s*(\d+)\s*días?"
    match = re.search(pattern, text, flags=re.IGNORECASE)
    if not match:
        return np.nan
    years, months, days = map(int, match.groups())
    total_days = years * 365.25 + months * 30.4375 + days
    return total_days / 30.4375


def clean_string(series):
    return (
        series.astype("string")
        .str.strip()
        .str.replace(r"\s+", " ", regex=True)
        .str.upper()
    )


def clean_who_category(series):
    out = clean_string(series)
    out = out.replace(
        {
            "NORMAL O MEDIANA": "Median",
            "NORMAL O  MEDIANA": "Median",
            "MEDIANA": "Median",
            "+1 DS": "+1 SD",
            "+2 DS": "+2 SD",
            "+3 DS": "+3 SD",
            "-1 DS": "-1 SD",
            "-2 DS": "-2 SD",
            "-3 DS": "-3 SD",
            "+1": "+1 SD",
            "+2": "+2 SD",
            "+3": "+3 SD",
            "-1": "-1 SD",
            "-2": "-2 SD",
            "-3": "-3 SD",
        }
    )
    return out


def yes_no(series):
    out = clean_string(series).replace({"SI": "Yes", "NO": "No"})
    return out.where(out.isin(["Yes", "No"]), pd.NA)


def format_p(p):
    if pd.isna(p):
        return ""
    if p < 0.001:
        return "<0.001"
    return f"{p:.3f}"


def save_figure(fig, stem):
    """Save the same publication figure as PNG and TIFF at 300 dpi."""
    for suffix in ["png", "tiff"]:
        fig.savefig(
            FIGURE_DIR / f"{stem}.{suffix}",
            dpi=DPI,
            bbox_inches="tight",
        )


def export_table(df, stem, sheet_name="Table"):
    df.to_csv(TABLE_DIR / f"{stem}.csv", index=False)
    df.to_excel(
        TABLE_DIR / f"{stem}.xlsx",
        index=False,
        sheet_name=sheet_name[:31],
    )


def bh_adjust(p_values):
    p_values = np.asarray(p_values, dtype=float)
    valid = np.isfinite(p_values)
    q = np.full(len(p_values), np.nan)
    if valid.any():
        q[valid] = multipletests(p_values[valid], method="fdr_bh")[1]
    return q


def model_or_table(model, drop_intercept=True, drop_spline=False):
    ci = model.conf_int()
    table = pd.DataFrame(
        {
            "Term": model.params.index,
            "log_OR": model.params.values,
            "OR": np.exp(model.params.values),
            "CI95_low": np.exp(ci.iloc[:, 0].values),
            "CI95_high": np.exp(ci.iloc[:, 1].values),
            "p_value": model.pvalues.values,
        }
    )
    if drop_intercept:
        table = table.loc[table["Term"] != "Intercept"].copy()
    if drop_spline:
        table = table.loc[~table["Term"].str.contains(r"bs\(", regex=True)].copy()
    table["q_BH"] = bh_adjust(table["p_value"].values)
    return table.reset_index(drop=True)


def global_wald_table(model):
    raw = model.wald_test_terms(skip_single=False).table.copy().reset_index()
    raw = raw.rename(columns={"index": "Term"})

    def scalar(value):
        try:
            return float(np.asarray(value).squeeze())
        except Exception:
            return np.nan

    for column in ["statistic", "pvalue", "df_constraint"]:
        if column in raw.columns:
            raw[column] = raw[column].map(scalar)
    if "pvalue" in raw.columns:
        raw = raw.rename(columns={"pvalue": "p_value"})
    return raw


def wilson_ci(events, n):
    low, high = proportion_confint(events, n, alpha=0.05, method="wilson")
    return float(low), float(high)


def pretty_term(term):
    replacements = {
        "C(sex, Treatment(reference='Male'))[T.Female]": "Female vs male",
        "C(year, Treatment(reference=2019))[T.2022]": "2022 vs 2019",
        "C(parental_figure, Treatment(reference='No'))[T.Yes]": "Father/other figure present vs no",
        "C(vulnerability, Treatment(reference='No'))[T.Yes]": "Recorded vulnerability: yes vs no",
        "C(vulnerability_status, Treatment(reference='No'))[T.Yes]": "Recorded vulnerability: yes vs no",
        "C(vulnerability_status, Treatment(reference='No'))[T.Missing]": "Vulnerability status missing vs no",
    }
    if term in replacements:
        return replacements[term]
    match = re.search(r"C\(height_for_age.*?\)\[T\.(.*?)\]", term)
    if match:
        return f"Height-for-age {match.group(1)} vs median"
    match = re.search(r"C\(weight_for_height.*?\)\[T\.(.*?)\]", term)
    if match:
        return f"Weight-for-height {match.group(1)} vs median"
    return term


def forest_plot(or_table, title, stem):
    plot_data = or_table.copy()
    plot_data = plot_data.loc[
        np.isfinite(plot_data["OR"])
        & np.isfinite(plot_data["CI95_low"])
        & np.isfinite(plot_data["CI95_high"])
        & (plot_data["OR"] > 0)
        & (plot_data["CI95_low"] > 0)
        & (plot_data["CI95_high"] > 0)
    ].copy()
    plot_data["Label"] = plot_data["Term"].map(pretty_term)
    plot_data = plot_data.iloc[::-1].reset_index(drop=True)

    y = np.arange(len(plot_data))
    xerr = np.vstack(
        [
            plot_data["OR"].values - plot_data["CI95_low"].values,
            plot_data["CI95_high"].values - plot_data["OR"].values,
        ]
    )

    fig_height = max(4.5, 0.42 * len(plot_data) + 1.8)
    fig, ax = plt.subplots(figsize=(8.5, fig_height))
    ax.errorbar(
        plot_data["OR"],
        y,
        xerr=xerr,
        fmt="o",
        capsize=3,
        linewidth=1.2,
    )
    ax.axvline(1.0, linestyle="--", linewidth=1.0)
    ax.set_xscale("log")
    ax.set_yticks(y)
    ax.set_yticklabels(plot_data["Label"])
    ax.set_xlabel("Adjusted odds ratio (95% CI)")
    ax.set_title(title)
    ax.grid(axis="x", alpha=0.25)
    fig.tight_layout()
    save_figure(fig, stem)
    plt.show()
    return fig

## 4. Load and prepare the analytical cohort

The original workbook is retained unchanged. The analysis uses only records with a valid DSM result. Children older than 60 months are excluded from the primary epidemiological cohort because the study focuses on the age range covered by the developmental assessment framework used in the source data.

In [ ]:
# Load source sheets
raw_2019 = pd.read_excel(DATA_PATH, sheet_name="reporte2019")
raw_2022 = pd.read_excel(DATA_PATH, sheet_name="reporte 2022")

required_columns = [TARGET] + list(SOURCE_COLUMNS.values())
for year, frame in [(2019, raw_2019), (2022, raw_2022)]:
    missing_columns = [col for col in required_columns if col not in frame.columns]
    if missing_columns:
        raise KeyError(f"Missing required columns in {year}: {missing_columns}")


def prepare_year(frame, year):
    out = frame.loc[frame[TARGET].isin(VALID_OUTCOMES)].copy()
    out["year"] = int(year)
    out["age_months"] = out[SOURCE_COLUMNS["age"]].apply(age_to_months)
    out["DSM_altered"] = (out[TARGET] != "Normal").astype(int)

    sex = clean_string(out[SOURCE_COLUMNS["sex"]])
    out["sex"] = sex.replace({"HOMBRE": "Male", "MUJER": "Female"})
    out["sex"] = out["sex"].where(out["sex"].isin(["Male", "Female"]), pd.NA)

    out["height_for_age"] = clean_who_category(out[SOURCE_COLUMNS["height_for_age"]])
    out["weight_for_height"] = clean_who_category(out[SOURCE_COLUMNS["weight_for_height"]])
    out["parental_figure"] = yes_no(out[SOURCE_COLUMNS["parental_figure"]])
    out["vulnerability"] = yes_no(out[SOURCE_COLUMNS["vulnerability"]])

    feeding = clean_string(out[SOURCE_COLUMNS["feeding"]])
    out["feeding_recorded"] = feeding.notna().map({True: "Recorded", False: "Missing"})
    out["feeding_pattern"] = feeding
    # Keep the source code L.M.E. separate; all other recorded patterns are grouped descriptively.
    out["feeding_LME"] = pd.Series(pd.NA, index=out.index, dtype="string")
    out.loc[feeding.notna(), "feeding_LME"] = "Other recorded pattern"
    out.loc[feeding.eq("L.M.E"), "feeding_LME"] = "L.M.E."

    return out


cohort_2019 = prepare_year(raw_2019, 2019)
cohort_2022 = prepare_year(raw_2022, 2022)
cohort_all = pd.concat([cohort_2019, cohort_2022], ignore_index=True)

analysis = cohort_all.loc[
    cohort_all["age_months"].between(AGE_MIN_MONTHS, AGE_MAX_MONTHS, inclusive="both")
].copy()

print(f"Raw records: 2019={len(raw_2019):,}; 2022={len(raw_2022):,}")
print(
    "Valid DSM records: "
    f"2019={len(cohort_2019):,}; 2022={len(cohort_2022):,}; total={len(cohort_all):,}"
)
print(f"Primary cohort (0–60 months): {len(analysis):,}")
print(f"Altered DSM events: {analysis['DSM_altered'].sum():,}")
print(f"Altered DSM prevalence: {analysis['DSM_altered'].mean() * 100:.2f}%")

## 5. Cohort flow and missingness audit

Missingness is reported explicitly because several contextual variables were recorded differently in 2019 and 2022. In particular, vulnerability is analyzed with complete-case, year-stratified, interaction, and missing-status sensitivity analyses rather than being silently imputed.

In [ ]:
# Cohort flow table
flow = pd.DataFrame(
    [
        ["2019 raw records", len(raw_2019)],
        ["2022 raw records", len(raw_2022)],
        ["2019 records with valid DSM outcome", len(cohort_2019)],
        ["2022 records with valid DSM outcome", len(cohort_2022)],
        ["Pooled records with valid DSM outcome", len(cohort_all)],
        ["Primary analytical cohort aged 0–60 months", len(analysis)],
        ["Normal DSM in primary cohort", int((analysis["DSM_altered"] == 0).sum())],
        ["Altered DSM in primary cohort", int((analysis["DSM_altered"] == 1).sum())],
    ],
    columns=["Stage", "N"],
)
export_table(flow, "Table_S1_Cohort_flow")
display(flow)

# Missingness audit for variables used in the publication analysis
selected_for_missingness = {
    "Sex": "sex",
    "Height-for-age": "height_for_age",
    "Weight-for-height": "weight_for_height",
    "Feeding pattern": "feeding_pattern",
    "Father/other figure at visit": "parental_figure",
    "Recorded vulnerability": "vulnerability",
}

missing_rows = []
for label, column in selected_for_missingness.items():
    for year in [2019, 2022]:
        subset = analysis.loc[analysis["year"] == year]
        missing_n = int(subset[column].isna().sum())
        missing_rows.append(
            {
                "Variable": label,
                "Year": year,
                "N": len(subset),
                "Missing_n": missing_n,
                "Missing_percent": 100 * missing_n / len(subset),
            }
        )

missingness = pd.DataFrame(missing_rows)
export_table(missingness, "Table_S2_Missingness_by_year")
display(missingness.round(2))

# Supplementary missingness figure
pivot_missing = missingness.pivot(index="Variable", columns="Year", values="Missing_percent")
fig, ax = plt.subplots(figsize=(8.5, 5.2))
pivot_missing.plot(kind="barh", ax=ax)
ax.set_xlabel("Missing data (%)")
ax.set_ylabel("")
ax.set_title("Missingness of selected analysis variables by year")
ax.legend(title="Year")
ax.grid(axis="x", alpha=0.25)
fig.tight_layout()
save_figure(fig, "Figure_S1_Missingness_by_year")
plt.show()

## 6. Table 1 — cohort characteristics by DSM status

Continuous age is summarized as median [IQR] and compared with a Mann–Whitney U test. Categorical variables are summarized as n (%) within each outcome group. Global categorical p-values are calculated on non-missing observations using Fisher's exact test for 2×2 tables with small expected counts and Pearson's chi-square test otherwise.

In [ ]:
def format_median_iqr(series):
    series = pd.to_numeric(series, errors="coerce").dropna()
    if len(series) == 0:
        return ""
    q1, med, q3 = series.quantile([0.25, 0.50, 0.75])
    return f"{med:.1f} [{q1:.1f}–{q3:.1f}]"


def categorical_global_p(df, variable):
    tmp = df[[variable, "DSM_altered"]].dropna()
    table = pd.crosstab(tmp[variable], tmp["DSM_altered"])
    if table.shape[0] < 2 or table.shape[1] < 2:
        return np.nan
    chi2, p_chi, dof, expected = chi2_contingency(table)
    if table.shape == (2, 2) and (expected < 5).any():
        _, p = fisher_exact(table.values)
        return p
    return p_chi


def table1_categorical_rows(df, variable, label):
    normal = df.loc[df["DSM_altered"] == 0]
    altered = df.loc[df["DSM_altered"] == 1]
    levels = [x for x in df[variable].dropna().astype(str).unique()]
    levels = sorted(levels)
    rows = []
    p = categorical_global_p(df, variable)

    for i, level in enumerate(levels):
        counts = []
        for subset in [df, normal, altered]:
            n = int(subset[variable].astype("string").eq(level).sum())
            counts.append(f"{n} ({100*n/len(subset):.1f}%)")
        rows.append(
            {
                "Variable": label if i == 0 else "",
                "Level": level,
                "Overall": counts[0],
                "Normal DSM": counts[1],
                "Altered DSM": counts[2],
                "p_value": format_p(p) if i == 0 else "",
            }
        )

    missing_counts = [int(x[variable].isna().sum()) for x in [df, normal, altered]]
    if any(missing_counts):
        rows.append(
            {
                "Variable": "",
                "Level": "Missing",
                "Overall": f"{missing_counts[0]} ({100*missing_counts[0]/len(df):.1f}%)",
                "Normal DSM": f"{missing_counts[1]} ({100*missing_counts[1]/len(normal):.1f}%)",
                "Altered DSM": f"{missing_counts[2]} ({100*missing_counts[2]/len(altered):.1f}%)",
                "p_value": "",
            }
        )
    return rows


table1_rows = []
normal = analysis.loc[analysis["DSM_altered"] == 0]
altered = analysis.loc[analysis["DSM_altered"] == 1]

age_p = mannwhitneyu(
    normal["age_months"].dropna(),
    altered["age_months"].dropna(),
    alternative="two-sided",
).pvalue

table1_rows.append(
    {
        "Variable": "Age, months",
        "Level": "Median [IQR]",
        "Overall": format_median_iqr(analysis["age_months"]),
        "Normal DSM": format_median_iqr(normal["age_months"]),
        "Altered DSM": format_median_iqr(altered["age_months"]),
        "p_value": format_p(age_p),
    }
)

for variable, label in [
    ("sex", "Sex"),
    ("year", "Assessment year"),
    ("height_for_age", "Height-for-age category"),
    ("weight_for_height", "Weight-for-height category"),
    ("feeding_LME", "Feeding pattern (source code L.M.E. vs other)"),
    ("parental_figure", "Father/other significant figure attended visit"),
    ("vulnerability", "Other vulnerability recorded"),
]:
    table1_rows.extend(table1_categorical_rows(analysis, variable, label))

Table1 = pd.DataFrame(table1_rows)
export_table(Table1, "Table_1_Cohort_characteristics")
display(Table1)

## 7. Core multivariable association model

The core model uses variables with high availability in both years and avoids sparse feeding categories that previously showed quasi-complete separation. Recorded vulnerability is analyzed separately because its missingness differs substantially by year.

**Core model:** altered DSM ~ nonlinear age + sex + year + height-for-age + weight-for-height + father/other significant figure attending the visit.

In [ ]:
core_variables = [
    "DSM_altered",
    "age_months",
    "sex",
    "year",
    "height_for_age",
    "weight_for_height",
    "parental_figure",
]
core_data = analysis[core_variables].dropna().copy()

CORE_FORMULA = """
DSM_altered ~
bs(age_months, df=5, degree=3, include_intercept=False)
+ C(sex, Treatment(reference='Male'))
+ C(year, Treatment(reference=2019))
+ C(height_for_age, Treatment(reference='Median'))
+ C(weight_for_height, Treatment(reference='Median'))
+ C(parental_figure, Treatment(reference='No'))
"""

core_model = smf.glm(
    formula=CORE_FORMULA,
    data=core_data,
    family=sm.families.Binomial(),
).fit(cov_type="HC3")

core_or = model_or_table(core_model, drop_spline=True)
core_or["Label"] = core_or["Term"].map(pretty_term)
core_global = global_wald_table(core_model)

export_table(core_or, "Table_2_Core_adjusted_ORs")
export_table(core_global, "Table_S3_Core_model_global_Wald_tests")

with open(MODEL_DIR / "core_model_summary.txt", "w", encoding="utf-8") as handle:
    handle.write(core_model.summary().as_text())

print(f"Core model N = {len(core_data):,}; altered DSM events = {core_data['DSM_altered'].sum():,}")
display(core_or[["Label", "OR", "CI95_low", "CI95_high", "p_value", "q_BH"]].round(4))
display(core_global)

forest_plot(
    core_or,
    title="Factors associated with altered psychomotor development",
    stem="Figure_1_Core_adjusted_OR_forest",
)

## 8. Recorded vulnerability — descriptive prevalence

Because vulnerability recording is incomplete and differs by year, prevalence is reported with Wilson 95% confidence intervals for the pooled cohort and separately for 2019 and 2022.

In [ ]:
prevalence_rows = []
for period_label, subset in [
    ("Overall", analysis),
    ("2019", analysis.loc[analysis["year"] == 2019]),
    ("2022", analysis.loc[analysis["year"] == 2022]),
]:
    for vulnerability_status in ["No", "Yes"]:
        group = subset.loc[subset["vulnerability"] == vulnerability_status]
        n = len(group)
        events = int(group["DSM_altered"].sum())
        if n == 0:
            continue
        low, high = wilson_ci(events, n)
        prevalence_rows.append(
            {
                "Period": period_label,
                "Recorded vulnerability": vulnerability_status,
                "N": n,
                "Altered_DSM_n": events,
                "Prevalence_percent": 100 * events / n,
                "CI95_low_percent": 100 * low,
                "CI95_high_percent": 100 * high,
            }
        )

vulnerability_prevalence = pd.DataFrame(prevalence_rows)
export_table(vulnerability_prevalence, "Table_3_Vulnerability_prevalence")
display(vulnerability_prevalence.round(2))

# Publication figure: prevalence by vulnerability status and year
period_order = ["Overall", "2019", "2022"]
x = np.arange(len(period_order), dtype=float)
offsets = {"No": -0.12, "Yes": 0.12}

fig, ax = plt.subplots(figsize=(7.8, 4.8))
for status in ["No", "Yes"]:
    sub = (
        vulnerability_prevalence.loc[
            vulnerability_prevalence["Recorded vulnerability"] == status
        ]
        .set_index("Period")
        .loc[period_order]
        .reset_index()
    )
    center = sub["Prevalence_percent"].values
    lower = center - sub["CI95_low_percent"].values
    upper = sub["CI95_high_percent"].values - center
    ax.errorbar(
        x + offsets[status],
        center,
        yerr=np.vstack([lower, upper]),
        fmt="o",
        capsize=4,
        linewidth=1.4,
        label=f"Vulnerability: {status.lower()}",
    )

ax.set_xticks(x)
ax.set_xticklabels(period_order)
ax.set_ylabel("Altered DSM prevalence (%)")
ax.set_xlabel("")
ax.set_title("Altered psychomotor development by recorded vulnerability")
ax.legend(frameon=False)
ax.grid(axis="y", alpha=0.25)
fig.tight_layout()
save_figure(fig, "Figure_2_Vulnerability_prevalence")
plt.show()

## 9. Recorded vulnerability — adjusted association and temporal heterogeneity

Three adjusted models estimate the vulnerability association in the pooled cohort, 2019, and 2022. A separate interaction model tests whether the vulnerability association differs between years.

These are **associations**, not causal effects. Differential missingness in vulnerability recording is explicitly treated as a limitation.

In [ ]:
VULNERABILITY_COVARIATES = [
    "DSM_altered",
    "age_months",
    "sex",
    "year",
    "height_for_age",
    "weight_for_height",
    "vulnerability",
]

vulnerability_cc = analysis[VULNERABILITY_COVARIATES].dropna().copy()

VULNERABILITY_FORMULA_POOLED = """
DSM_altered ~
bs(age_months, df=5, degree=3, include_intercept=False)
+ C(sex, Treatment(reference='Male'))
+ C(year, Treatment(reference=2019))
+ C(height_for_age, Treatment(reference='Median'))
+ C(weight_for_height, Treatment(reference='Median'))
+ C(vulnerability, Treatment(reference='No'))
"""

vulnerability_term = "C(vulnerability, Treatment(reference='No'))[T.Yes]"

vulnerability_models = []

# Pooled model
pooled_vulnerability_model = smf.glm(
    formula=VULNERABILITY_FORMULA_POOLED,
    data=vulnerability_cc,
    family=sm.families.Binomial(),
).fit(cov_type="HC3")

# Year-stratified models
STRATIFIED_FORMULA = """
DSM_altered ~
bs(age_months, df=5, degree=3, include_intercept=False)
+ C(sex, Treatment(reference='Male'))
+ C(height_for_age, Treatment(reference='Median'))
+ C(weight_for_height, Treatment(reference='Median'))
+ C(vulnerability, Treatment(reference='No'))
"""

for label, data_subset, model in [
    ("Overall", vulnerability_cc, pooled_vulnerability_model),
    (
        "2019",
        vulnerability_cc.loc[vulnerability_cc["year"] == 2019],
        smf.glm(
            formula=STRATIFIED_FORMULA,
            data=vulnerability_cc.loc[vulnerability_cc["year"] == 2019],
            family=sm.families.Binomial(),
        ).fit(cov_type="HC3"),
    ),
    (
        "2022",
        vulnerability_cc.loc[vulnerability_cc["year"] == 2022],
        smf.glm(
            formula=STRATIFIED_FORMULA,
            data=vulnerability_cc.loc[vulnerability_cc["year"] == 2022],
            family=sm.families.Binomial(),
        ).fit(cov_type="HC3"),
    ),
]:
    beta = model.params[vulnerability_term]
    ci = model.conf_int().loc[vulnerability_term]
    vulnerability_models.append(
        {
            "Period": label,
            "N": len(data_subset),
            "Altered_DSM_n": int(data_subset["DSM_altered"].sum()),
            "Adjusted_OR": float(np.exp(beta)),
            "CI95_low": float(np.exp(ci.iloc[0])),
            "CI95_high": float(np.exp(ci.iloc[1])),
            "p_value": float(model.pvalues[vulnerability_term]),
        }
    )

vulnerability_or = pd.DataFrame(vulnerability_models)

# Interaction model
INTERACTION_FORMULA = """
DSM_altered ~
bs(age_months, df=5, degree=3, include_intercept=False)
+ C(sex, Treatment(reference='Male'))
+ C(height_for_age, Treatment(reference='Median'))
+ C(weight_for_height, Treatment(reference='Median'))
+ C(vulnerability, Treatment(reference='No'))
* C(year, Treatment(reference=2019))
"""

interaction_model = smf.glm(
    formula=INTERACTION_FORMULA,
    data=vulnerability_cc,
    family=sm.families.Binomial(),
).fit(cov_type="HC3")

interaction_term = (
    "C(vulnerability, Treatment(reference='No'))[T.Yes]:"
    "C(year, Treatment(reference=2019))[T.2022]"
)
interaction_beta = interaction_model.params[interaction_term]
interaction_ci = interaction_model.conf_int().loc[interaction_term]
interaction_result = pd.DataFrame(
    [
        {
            "Contrast": "Vulnerability association: 2022 vs 2019 (ratio of ORs)",
            "Ratio_of_ORs": float(np.exp(interaction_beta)),
            "CI95_low": float(np.exp(interaction_ci.iloc[0])),
            "CI95_high": float(np.exp(interaction_ci.iloc[1])),
            "p_value": float(interaction_model.pvalues[interaction_term]),
        }
    ]
)

export_table(vulnerability_or, "Table_4_Vulnerability_adjusted_association")
export_table(interaction_result, "Table_5_Vulnerability_year_interaction")

with open(MODEL_DIR / "vulnerability_pooled_model_summary.txt", "w", encoding="utf-8") as handle:
    handle.write(pooled_vulnerability_model.summary().as_text())
with open(MODEL_DIR / "vulnerability_interaction_model_summary.txt", "w", encoding="utf-8") as handle:
    handle.write(interaction_model.summary().as_text())

print("Adjusted vulnerability association")
display(vulnerability_or.round(4))
print("Vulnerability × year interaction")
display(interaction_result.round(4))

# Forest plot of adjusted vulnerability ORs
forest_data = vulnerability_or.rename(
    columns={
        "Period": "Term",
        "Adjusted_OR": "OR",
    }
).copy()
forest_data["CI95_low"] = vulnerability_or["CI95_low"]
forest_data["CI95_high"] = vulnerability_or["CI95_high"]
forest_data["Term"] = forest_data["Term"].map(
    {
        "Overall": "Pooled 2019 + 2022",
        "2019": "2019",
        "2022": "2022",
    }
)

fig, ax = plt.subplots(figsize=(7.5, 4.2))
plot_data = forest_data.iloc[::-1].reset_index(drop=True)
y = np.arange(len(plot_data))
xerr = np.vstack(
    [
        plot_data["OR"].values - plot_data["CI95_low"].values,
        plot_data["CI95_high"].values - plot_data["OR"].values,
    ]
)
ax.errorbar(plot_data["OR"], y, xerr=xerr, fmt="o", capsize=4, linewidth=1.4)
ax.axvline(1.0, linestyle="--", linewidth=1.0)
ax.set_xscale("log")
ax.set_yticks(y)
ax.set_yticklabels(plot_data["Term"])
ax.set_xlabel("Adjusted odds ratio for altered DSM (95% CI)")
ax.set_title("Association of recorded vulnerability with altered psychomotor development")
ax.grid(axis="x", alpha=0.25)
fig.tight_layout()
save_figure(fig, "Figure_3_Vulnerability_adjusted_OR_forest")
plt.show()

## 10. Sensitivity analysis for missing vulnerability status

The complete-case vulnerability analysis may be affected by non-random recording. This sensitivity model retains missing vulnerability as an explicit category. It does **not** solve missing-not-at-random bias, but it shows whether missing records behave differently from explicitly recorded `No` records after adjustment.

In [ ]:
sensitivity = analysis[
    [
        "DSM_altered",
        "age_months",
        "sex",
        "year",
        "height_for_age",
        "weight_for_height",
        "vulnerability",
    ]
].copy()

sensitivity["vulnerability_status"] = sensitivity["vulnerability"].astype("string")
sensitivity["vulnerability_status"] = sensitivity["vulnerability_status"].fillna("Missing")
sensitivity = sensitivity.dropna(
    subset=["DSM_altered", "age_months", "sex", "year", "height_for_age", "weight_for_height"]
)

SENSITIVITY_FORMULA = """
DSM_altered ~
bs(age_months, df=5, degree=3, include_intercept=False)
+ C(sex, Treatment(reference='Male'))
+ C(year, Treatment(reference=2019))
+ C(height_for_age, Treatment(reference='Median'))
+ C(weight_for_height, Treatment(reference='Median'))
+ C(vulnerability_status, Treatment(reference='No'))
"""

sensitivity_model = smf.glm(
    formula=SENSITIVITY_FORMULA,
    data=sensitivity,
    family=sm.families.Binomial(),
).fit(cov_type="HC3")

sensitivity_or = model_or_table(sensitivity_model, drop_spline=True)
sensitivity_or["Label"] = sensitivity_or["Term"].map(pretty_term)
export_table(sensitivity_or, "Table_S4_Vulnerability_missing_status_sensitivity")

with open(MODEL_DIR / "vulnerability_missing_status_sensitivity_summary.txt", "w", encoding="utf-8") as handle:
    handle.write(sensitivity_model.summary().as_text())

display(
    sensitivity_or[["Label", "OR", "CI95_low", "CI95_high", "p_value", "q_BH"]].round(4)
)

## 11. Automated publication summary and analysis manifest

This cell writes a concise machine-readable manifest and a text summary of the main findings. The text is intended as a reproducibility aid, not as a substitute for manuscript interpretation.

In [ ]:
# Pull key estimates directly from fitted models
female_term = "C(sex, Treatment(reference='Male'))[T.Female]"
year_term = "C(year, Treatment(reference=2019))[T.2022]"
parent_term = "C(parental_figure, Treatment(reference='No'))[T.Yes]"


def estimate_triplet(model, term):
    beta = model.params[term]
    ci = model.conf_int().loc[term]
    return {
        "OR": float(np.exp(beta)),
        "CI95_low": float(np.exp(ci.iloc[0])),
        "CI95_high": float(np.exp(ci.iloc[1])),
        "p_value": float(model.pvalues[term]),
    }

key_results = {
    "primary_cohort_N": int(len(analysis)),
    "altered_DSM_n": int(analysis["DSM_altered"].sum()),
    "altered_DSM_prevalence": float(analysis["DSM_altered"].mean()),
    "core_model_N": int(len(core_data)),
    "female_vs_male": estimate_triplet(core_model, female_term),
    "year_2022_vs_2019": estimate_triplet(core_model, year_term),
    "parental_figure_yes_vs_no": estimate_triplet(core_model, parent_term),
    "vulnerability_pooled": vulnerability_or.loc[
        vulnerability_or["Period"] == "Overall"
    ].iloc[0].to_dict(),
    "vulnerability_2019": vulnerability_or.loc[
        vulnerability_or["Period"] == "2019"
    ].iloc[0].to_dict(),
    "vulnerability_2022": vulnerability_or.loc[
        vulnerability_or["Period"] == "2022"
    ].iloc[0].to_dict(),
    "vulnerability_year_interaction": interaction_result.iloc[0].to_dict(),
}

with open(OUTPUT_DIR / "analysis_manifest.json", "w", encoding="utf-8") as handle:
    json.dump(key_results, handle, ensure_ascii=False, indent=2)

pooled = key_results["vulnerability_pooled"]
interaction = key_results["vulnerability_year_interaction"]
summary_lines = [
    "EPIDEMIOLOGICAL ANALYSIS SUMMARY",
    "=" * 72,
    f"Primary cohort: N={key_results['primary_cohort_N']:,}, altered DSM={key_results['altered_DSM_n']:,} "
    f"({100*key_results['altered_DSM_prevalence']:.2f}%).",
    "",
    "Core adjusted model:",
    f"- Female vs male: OR {key_results['female_vs_male']['OR']:.2f} "
    f"(95% CI {key_results['female_vs_male']['CI95_low']:.2f}–{key_results['female_vs_male']['CI95_high']:.2f}), "
    f"p={key_results['female_vs_male']['p_value']:.4g}.",
    f"- 2022 vs 2019: OR {key_results['year_2022_vs_2019']['OR']:.2f} "
    f"(95% CI {key_results['year_2022_vs_2019']['CI95_low']:.2f}–{key_results['year_2022_vs_2019']['CI95_high']:.2f}), "
    f"p={key_results['year_2022_vs_2019']['p_value']:.4g}.",
    f"- Father/other figure present vs no: OR {key_results['parental_figure_yes_vs_no']['OR']:.2f} "
    f"(95% CI {key_results['parental_figure_yes_vs_no']['CI95_low']:.2f}–{key_results['parental_figure_yes_vs_no']['CI95_high']:.2f}), "
    f"p={key_results['parental_figure_yes_vs_no']['p_value']:.4g}.",
    "",
    "Recorded vulnerability analysis:",
    f"- Pooled adjusted OR {float(pooled['Adjusted_OR']):.2f} "
    f"(95% CI {float(pooled['CI95_low']):.2f}–{float(pooled['CI95_high']):.2f}), "
    f"p={float(pooled['p_value']):.4g}.",
    f"- Vulnerability × year ratio of ORs {float(interaction['Ratio_of_ORs']):.2f} "
    f"(95% CI {float(interaction['CI95_low']):.2f}–{float(interaction['CI95_high']):.2f}), "
    f"p={float(interaction['p_value']):.4g}.",
    "",
    "Interpretation note: all estimates are associations, not causal effects. Differential vulnerability missingness by year must be reported as a limitation.",
]

summary_text = "\n".join(summary_lines)
print(summary_text)
with open(OUTPUT_DIR / "key_results_summary.txt", "w", encoding="utf-8") as handle:
    handle.write(summary_text)

## 12. Output checklist

After a complete run, the output directory contains:

### Main tables
- `Table_1_Cohort_characteristics`
- `Table_2_Core_adjusted_ORs`
- `Table_3_Vulnerability_prevalence`
- `Table_4_Vulnerability_adjusted_association`
- `Table_5_Vulnerability_year_interaction`

### Main figures
- `Figure_1_Core_adjusted_OR_forest` (PNG and TIFF, 300 dpi)
- `Figure_2_Vulnerability_prevalence` (PNG and TIFF, 300 dpi)
- `Figure_3_Vulnerability_adjusted_OR_forest` (PNG and TIFF, 300 dpi)

### Supplementary outputs
- Cohort flow
- Missingness by year
- Core-model global Wald tests
- Vulnerability missing-status sensitivity analysis
- Full text summaries of fitted models
- Package and Python versions
- `analysis_manifest.json`

### Reporting cautions
1. Report associations, not causal effects.
2. Explicitly report the large difference in vulnerability missingness between years.
3. Prefer global Wald tests for multi-level anthropometric variables instead of highlighting a single category solely because its coefficient has `p < 0.05`.
4. The vulnerability association should be described as applying to children with recorded vulnerability information, with sensitivity analyses used to assess missing-status behavior.